# Daily demand report — 2022 vs 2023 (fixed)

Corrected version of `mock_05`. Each fix is marked with **Fix N**, numbered as in
`mock_05_solution.md`.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

raw = pd.read_csv("../../data/hourly_power_raw.csv")
raw.shape

(17457, 7)

**Fix 1** — the feed is UTC (the clean reference file carries `+00:00`; a local feed
would have a missing 01:00 on the last Sunday of March and a duplicated one in October).
Parse with `utc=True`, and derive local time explicitly when needed.

**Fix 3** — drop the 15 duplicated rows before doing anything with the time axis.

In [2]:
df = raw.copy()
df["time"] = pd.to_datetime(df["time"], utc=True)
print("exact duplicate rows :", df.duplicated().sum())
print("duplicate timestamps :", df.duplicated("time").sum())
df = df.drop_duplicates().drop_duplicates("time", keep="last")
df = df.set_index("time").sort_index()
df = df.drop(columns="region")  # constant column
df.shape

exact duplicate rows : 15
duplicate timestamps : 15


(17442, 5)

**Fix 4** — `replace("missing", NaN)` leaves the column as `object`; use `pd.to_numeric(errors="coerce")`
and *leave the gaps as NaN*. Filling price with 0 is a made-up observation (0 is a plausible
price in this market — there are negative prices — so nothing would flag it later).

**Fix 8** — `-999` is a sentinel, not a temperature.

In [3]:
df["price_eur_mwh"] = pd.to_numeric(df["price_eur_mwh"], errors="coerce")
print("price NaN after coercion:", df["price_eur_mwh"].isna().sum())
print("temp sentinels          :", (df["temp_c"] == -999).sum())
df.loc[df["temp_c"] == -999, "temp_c"] = np.nan
df.describe().round(1)

price NaN after coercion: 100
temp sentinels          : 63


,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh
count,17442.0,17230.0,17442.0,17442.0,17342.0
mean,29314.9,9.9,7.3,97.7,98.5
std,4207.4,6.7,2.5,155.3,36.9
min,18092.9,-6.4,0.0,0.0,-19.9
25%,26444.7,4.5,5.6,0.0,73.5
50%,29670.6,10.0,7.2,0.0,97.7
75%,32372.8,15.3,9.0,143.1,122.8
max,40824.9,27.7,16.0,794.6,419.6


**Fix 2 / 9** — complete the hourly grid but do *not* forward-fill the measurements. Missing hours
stay NaN and are counted, so that daily totals from incomplete days can be excluded rather than
reported as low.

In [4]:
full = pd.date_range(df.index.min(), df.index.max(), freq="h", tz="UTC")
hourly = df.reindex(full)
missing = hourly["consumption_mwh"].isna()
print("missing hours:", missing.sum())
print(missing[missing].groupby(missing[missing].index.date).size().sort_values(ascending=False).head(3))

missing hours: 78
2022-03-27    24
2022-06-07     2
2022-01-07     1
Name: consumption_mwh, dtype: int64


**Fix 1 (cont.) / 6 / 11** — everything the report calls "local" (day boundaries, peak hour,
17–20 evening window) is computed on a `Europe/London` index. Local days have 23, 24 or 25 hours.

In [5]:
local = hourly.tz_convert("Europe/London")
grp = local["consumption_mwh"].resample("D")

hours_expected = local.index.to_series().resample("D").size()   # 23 / 24 / 25
hours_present = grp.count()

daily = pd.DataFrame({
    "energy_mwh": grp.sum(min_count=1),
    "hours_expected": hours_expected,
    "hours_present": hours_present,
})
daily["complete"] = daily["hours_present"] == daily["hours_expected"]
print("incomplete days:", (~daily["complete"]).sum())
daily["hours_expected"].value_counts()

incomplete days: 55


hours_expected
24    726
23      2
25      2
Name: count, dtype: int64

**Fix 7** — default `label="left"`: the total for 1 Jan is stamped 1 Jan. Lowest demand day is now
searched among *complete* days only.

In [6]:
daily.loc[daily["complete"]].nsmallest(5, "energy_mwh")[["energy_mwh", "hours_present"]]

,energy_mwh,hours_present
2022-06-26 00:00:00+01:00,594988.1,24
2023-08-27 00:00:00+01:00,595066.1,24
2022-06-19 00:00:00+01:00,597875.9,24
2023-07-09 00:00:00+01:00,597942.8,24
2022-07-17 00:00:00+01:00,598569.9,24


**Fix 5** — "average price paid" is the load-weighted price, not the simple mean of hourly prices.
Hours where either price or load is missing are excluded from both numerator and denominator.

In [7]:
ok = local["price_eur_mwh"].notna() & local["consumption_mwh"].notna()
cost = (local["price_eur_mwh"] * local["consumption_mwh"]).where(ok)
load = local["consumption_mwh"].where(ok)
daily["avg_price_simple"] = local["price_eur_mwh"].resample("D").mean()
daily["avg_price_paid"] = cost.resample("D").sum(min_count=1) / load.resample("D").sum(min_count=1)
daily["min_temp"] = local["temp_c"].resample("D").min()
daily["mean_temp"] = local["temp_c"].resample("D").mean()
daily[["avg_price_simple", "avg_price_paid", "min_temp"]].describe().round(2)

,avg_price_simple,avg_price_paid,min_temp
count,729.00,729.00,729.00
mean,98.50,100.68,6.26
std,25.97,25.94,6.34
min,26.94,28.23,-6.40
25%,78.22,80.37,0.60
50%,99.01,100.82,6.18
75%,117.43,119.86,11.84
max,174.23,177.21,20.70


In [8]:
daily.nsmallest(3, "min_temp")[["min_temp", "mean_temp", "complete"]]

,min_temp,mean_temp,complete
2023-01-24 00:00:00+00:00,-6.40,-2.864348,True
2022-01-31 00:00:00+00:00,-6.21,-1.811250,True
2023-01-25 00:00:00+00:00,-6.20,-2.999167,True


**Fix 6 / 9** — peak hour on local time, NaN where the day has no data (no forward-filled fake days).

In [9]:
def peak_hour(x):
    return x.idxmax().hour if x.notna().any() else np.nan

daily["peak_hour"] = grp.agg(peak_hour)
daily.groupby(daily.index.month)["peak_hour"].agg(lambda s: s.mode()[0])

1     18.0
2     18.0
3     18.0
4     19.0
5     19.0
6     19.0
7     19.0
8     19.0
9     19.0
10    19.0
11    18.0
12    18.0
Name: peak_hour, dtype: float64

**Fix 11** — evening window on local time, same denominator as the daily total, complete days only.

In [10]:
evening = local.between_time("17:00", "20:00")["consumption_mwh"].resample("D").sum(min_count=1)
daily["evening_share"] = (evening / daily["energy_mwh"]).where(daily["complete"])
daily["evening_share"].describe().round(3)

count    675.000
mean       0.193
std        0.005
min        0.180
25%        0.190
50%        0.193
75%        0.197
max        0.207
Name: evening_share, dtype: float64

**Fix 10** — align years on the calendar date (month-day), not on array position, and only compare
days that are complete in both years. Report the aggregate ratio as well as the mean of daily ratios.

In [11]:
d2022 = daily.loc["2022"].copy()
d2023 = daily.loc["2023"].copy()
d2022.index = d2022.index.strftime("%m-%d")
d2023.index = d2023.index.strftime("%m-%d")
pair = d2022[["energy_mwh", "complete"]].join(d2023[["energy_mwh", "complete"]], lsuffix="_22", rsuffix="_23")
pair = pair.loc[pair["complete_22"] & pair["complete_23"]]
print("paired complete days:", len(pair))
yoy_daily = (pair["energy_mwh_23"] / pair["energy_mwh_22"] - 1)
yoy_total = pair["energy_mwh_23"].sum() / pair["energy_mwh_22"].sum() - 1
print(f"mean of daily YoY ratios : {yoy_daily.mean()*100:+.2f} %")
print(f"aggregate YoY over pairs : {yoy_total*100:+.2f} %")

paired complete days: 312
mean of daily YoY ratios : -0.41 %
aggregate YoY over pairs : -0.57 %


## Results (honest)

In [12]:
comp = daily.loc[daily["complete"]]
lowest = comp["energy_mwh"].idxmin()
coldest = daily["min_temp"].idxmin()
peak_by_season = daily.groupby(daily.index.month)["peak_hour"].agg(lambda s: s.mode()[0])
p22 = daily.loc["2022"]; p23 = daily.loc["2023"]

print(f"Lowest demand day (complete days): {lowest.date()}  ({comp.loc[lowest, 'energy_mwh']:,.0f} MWh)")
print(f"Coldest day                      : {coldest.date()}  (min temp {daily.loc[coldest, 'min_temp']:.1f} C)")
print(f"Average price paid 2022 (load-wt): {p22['avg_price_paid'].mean():.2f} EUR/MWh   simple mean {p22['avg_price_simple'].mean():.2f}")
print(f"Average price paid 2023 (load-wt): {p23['avg_price_paid'].mean():.2f} EUR/MWh   simple mean {p23['avg_price_simple'].mean():.2f}")
print(f"YoY change in daily energy       : {yoy_total*100:+.2f} % (aggregate), {yoy_daily.mean()*100:+.2f} % (mean of daily)")
print(f"Evening (17-20 local) share      : {daily['evening_share'].mean()*100:.2f} %")
print(f"Typical peak hour (local)        : Jan {peak_by_season[1]:.0f}h, Jul {peak_by_season[7]:.0f}h")

Lowest demand day (complete days): 2022-06-26  (594,988 MWh)
Coldest day                      : 2023-01-24  (min temp -6.4 C)
Average price paid 2022 (load-wt): 114.46 EUR/MWh   simple mean 112.31
Average price paid 2023 (load-wt): 86.94 EUR/MWh   simple mean 84.72
YoY change in daily energy       : -0.57 % (aggregate), -0.41 % (mean of daily)
Evening (17-20 local) share      : 19.33 %
Typical peak hour (local)        : Jan 18h, Jul 19h
